# Projet : Contamination des sols à la chlordécone en Martinique

Younes Ouloum - ENSAR A1 Data Science  
Mars 2026 - UE Ingénierie des données (S. Manou-Abi)

Ce notebook constitue le livrable principal du projet. Il couvre les deux volets demandés :
- **Volet 1** : ingénierie des données (nettoyage, transformation, jointures, flux, dates, texte, spatial)
- **Volet 2** : analyse statistique (EDA, ACP, AFC, clustering, KNN, tests, aide à la décision)

Le dataset `BaseCLD2026.csv` contient 31 126 mesures de contamination à la chlordécone sur des parcelles agricoles martiniquaises, collectées entre 2010 et 2019.


## 1. Chargement des données et premieres observations

On commence par importer les librairies et charger le fichier CSV. Le séparateur est le point-virgule (format français) et l'encodage est UTF-8. On corrige aussi les caracteres mal encodés dans `type_sol` (un probleme de double encodage fait apparaitre des symboles comme `…` et `‚` a la place des accents).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import chi2_contingency, f_oneway, spearmanr, shapiro, kruskal, ks_2samp
import warnings

# on filtre uniquement les FutureWarning de sklearn, pas tous les warnings
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")

# seed global pour la reproductibilite
np.random.seed(42)


In [ ]:
df = pd.read_csv("../data/BaseCLD2026.csv", sep=";", encoding="utf-8")

# correction des caracteres mal encodes dans type_sol (probleme de double encodage)
df["type_sol"] = df["type_sol"].str.replace("\u2026", "...", regex=False)
df["type_sol"] = df["type_sol"].str.replace("\u201a", "e", regex=False)
print("Taille du dataset :", df.shape)
print()
print("Colonnes :")
for i, col in enumerate(df.columns, 1):
    print("  {}. {}".format(i, col))


In [ ]:
df.head()


In [ ]:
df.dtypes


In [ ]:
df.describe()


**Premieres remarques** : le dataset a 31 126 lignes et 22 colonnes. On note que `Taux_5b_hydro` est de type `object` alors que c'est censé etre numérique (il y a des virgules francaises, des "inf" et des "999" dedans). `histoBanane_Histo_ban` a beaucoup de NaN. On va traiter tout ca dans la partie nettoyage.


### Structure des donnees : attention a la granularite

Chaque `ID` represente une **parcelle agricole**. Mais une parcelle peut avoir **plusieurs lignes** dans le dataset : ce sont des points raster (grille MNT) a l'interieur de la parcelle. Le taux de chlordecone est le meme pour tous les points d'une parcelle, mais les variables topographiques (pente, rugosite, ombrage...) varient d'un point a l'autre.

Consequence : il y a 31 126 lignes mais seulement ~3 600 parcelles uniques. Pour les analyses sur le taux de CLD, on travaillera parfois sur les donnees agregees par parcelle pour eviter la pseudo-replication (surestimation de la significativite).


## 2. Manipulation de données tabulaires

Cette section montre les opérations fondamentales de manipulation de dataframes avec pandas. En R on utiliserait dplyr avec les verbes `group_by`, `summarise`, `filter`, `mutate`, `arrange`. En Python, pandas offre les memes fonctionnalités.


### 2.1 Agrégation par commune (group_by + summarise)

On calcule le taux moyen, médian et max de chlordécone par commune pour avoir une vue d'ensemble de la contamination.


In [ ]:
resume_commune = (
    df.groupby("COMMU_LAB")
    .agg(
        nb_parcelles=("ID", "count"),
        taux_moyen=("Taux_Chlordecone", "mean"),
        taux_median=("Taux_Chlordecone", "median"),
        taux_max=("Taux_Chlordecone", "max"),
    )
    .reset_index()
    .sort_values("taux_moyen", ascending=False)
)

resume_commune.head(10)


Les communes du nord de la Martinique (Gros-Morne, Basse-Pointe, Macouba) affichent les taux moyens les plus élevés. Ca correspond aux zones historiques de culture bananière.


### 2.2 Filtrage (filter)

On isole les parcelles fortement contaminées (taux > 1 mg/kg) et on combine plusieurs critères de filtrage.


In [ ]:
# filtre simple
df_forte = df[df["Taux_Chlordecone"] > 1]
print("Parcelles > 1 mg/kg :", len(df_forte), "soit {:.1f}%".format(len(df_forte)/len(df)*100))

# filtre combiné : Andosol + forte contamination + données recentes
df_filtre = df[
    (df["Taux_Chlordecone"] > 0.5) &
    (df["Sol_simple"] == "Andosol") &
    (df["ANNEE"] >= 2015)
]
print("Andosol + >0.5 mg/kg + apres 2015 :", len(df_filtre), "parcelles")


### 2.3 Création de variables (mutate)

On crée plusieurs variables dérivées qui seront utiles pour la suite de l'analyse.


In [ ]:
# classes de contamination
df["classe_contam"] = pd.cut(
    df["Taux_Chlordecone"],
    bins=[0, 0.01, 0.1, 1, 5, 20],
    labels=["Tres faible", "Faible", "Modere", "Eleve", "Tres eleve"],
    include_lowest=True
)

# transformation log (la distribution est tres asymétrique, le log la normalise)
df["log_taux"] = np.log1p(df["Taux_Chlordecone"])

# indicateur : est-ce que la chlordécone a été détectée ou est sous le seuil ?
# Operateur_chld = "=" signifie valeur exacte, "<" signifie sous le seuil de detection
df["detecte"] = (df["Operateur_chld"] == "=")

print("Classes de contamination :")
print(df["classe_contam"].value_counts().sort_index())
print()
print("Taux de detection :", df["detecte"].mean()*100, "%")


In [ ]:
# tri : les 10 parcelles les plus contaminées
df.nlargest(10, "Taux_Chlordecone")[["ID", "COMMU_LAB", "ANNEE", "Taux_Chlordecone", "Sol_simple"]]


### 2.4 Détection des incohérences

Avant de nettoyer les données, il faut repérer tous les problemes. On cherche systematiquement les valeurs aberrantes, manquantes ou incohérentes.


In [ ]:
print("=== Bilan des incohérences ===")
print()

# dates aberrantes
n_date_9999 = df["Date_analyse"].str.contains("9999", na=False).sum()
print("1. Dates d'analyse avec annee 9999 :", n_date_9999)

# Taux_5b_hydro : valeurs aberrantes
n_inf = (df["Taux_5b_hydro"].astype(str).str.strip() == "inf").sum()
n_999 = (df["Taux_5b_hydro"].astype(str).str.strip() == "999").sum()
print("2. Taux_5b_hydro = 'inf' :", n_inf)
print("   Taux_5b_hydro = '999' :", n_999)

# communes manquantes
print("3. Communes manquantes :", df["COMMU_LAB"].isna().sum())

# hors martinique
print("4. 'Hors Martinique' :", (df["COMMU_LAB"] == "Hors Martinique").sum())

# sol = No data ou Urban area
print("5. Sol = 'No data' ou 'Urban area' :", df["Sol_simple"].isin(["No data", "Urban area"]).sum())

# dates incohérentes
dp = pd.to_datetime(df["Date_prelevement"], format="%d/%m/%Y", errors="coerce")
da = pd.to_datetime(df["Date_analyse"], format="%d/%m/%Y", errors="coerce")
print("6. Analyse avant prelevement :", (da < dp).sum())

# taux négatifs
print("7. Taux negatifs :", (df["Taux_Chlordecone"] < 0).sum())


On a trouvé pas mal de problemes : 2 702 dates d'analyse à "9999" (probablement des valeurs manquantes codées comme ca), des valeurs "inf" et "999" dans Taux_5b_hydro, 298 communes manquantes, 27 parcelles hors Martinique. Tout ca sera traité dans la section nettoyage.


In [ ]:
# nettoyage des noms de communes : on retire les parentheses
df["commune_clean"] = df["COMMU_LAB"].str.replace(r"\(.*\)", "", regex=True).str.strip()

# suppression des lignes hors Martinique
n_avant = len(df)
df = df[df["COMMU_LAB"] != "Hors Martinique"].copy()
print("Suppression 'Hors Martinique' :", n_avant, "->", len(df), "lignes")

# indexation : acceder a une valeur precise
print("Taux de la ligne 0 :", df.iloc[0]["Taux_Chlordecone"])
print("Communes de la parcelle 20143 :", df.loc[df["ID"]==20143, "COMMU_LAB"].unique())


## 3. Recodage et transformation de modalités

On recode certaines variables pour les rendre plus exploitables dans l'analyse.


In [ ]:
# recodage de la pluviometrie en categories lisibles
df["pluie_cat"] = df["RAIN"].map({
    "0-1250": "Faible",
    "1250-1500": "Moderee",
    "1500-2000": "Moderee-haute",
    "2000-3000": "Haute",
    "3000-5000": "Tres haute",
    "5000-8000": "Extreme"
})

# recodage de l'historique bananier
df["histo_banane"] = df["histoBanane_Histo_ban"].map({
    1.0: "Jamais banane",
    2.0: "Banane passee",
    3.0: "Banane actuelle"
})

print("Pluviometrie recodee :")
print(df["pluie_cat"].value_counts())
print()
print("Historique bananier :")
print(df["histo_banane"].value_counts(dropna=False))


In [ ]:
# extraction de sous-ensembles
df_andosol = df[df["Sol_simple"] == "Andosol"]
df_recent = df[df["ANNEE"].between(2015, 2019)]
print("Sous-ensemble Andosol :", len(df_andosol))
print("Sous-ensemble 2015-2019 :", len(df_recent))

# agregation croisee commune x sol
agg = (
    df.groupby(["COMMU_LAB", "Sol_simple"])
    .agg(taux_moyen=("Taux_Chlordecone", "mean"), n=("ID", "count"))
    .reset_index()
    .sort_values("taux_moyen", ascending=False)
)
agg.head(10)


## 4. Jointures de données

Les jointures permettent de combiner des tables. En SQL et en R (dplyr) on parle de `left_join`, `right_join`, `inner_join`, `full_join`. Pandas utilise `pd.merge()` avec le parametre `how`.

On crée une table auxiliaire avec des infos géographiques pour illustrer les 4 types de jointures et leur impact.


In [ ]:
# table auxiliaire
communes_info = pd.DataFrame({
    "COMMU_LAB": ["GROS-MORNE", "DUCOS", "SAINTE-MARIE", "LORRAIN(LE)",
                  "ROBERT(LE)", "SAINT-JOSEPH", "LAMENTIN(LE)", "COMMUNE_FICTIVE"],
    "zone_geo": ["Centre", "Sud", "Nord-Atl.", "Nord-Atl.",
                 "Centre-Atl.", "Centre-int.", "Centre", "Nord"],
    "pop_2020": [10200, 18500, 15800, 7200, 22300, 16200, 40400, 999]
})

# resume par commune
resume = df.groupby("COMMU_LAB").agg(
    taux_moyen=("Taux_Chlordecone", "mean"), n=("ID", "count")
).reset_index()

# les 4 jointures
left = pd.merge(resume, communes_info, on="COMMU_LAB", how="left")
right = pd.merge(resume, communes_info, on="COMMU_LAB", how="right")
inner = pd.merge(resume, communes_info, on="COMMU_LAB", how="inner")
outer = pd.merge(resume, communes_info, on="COMMU_LAB", how="outer")

print("{:<12} {:<8} {:<12} {:<12}".format("Jointure", "Lignes", "NaN gauche", "NaN droite"))
for nom, res in [("LEFT", left), ("RIGHT", right), ("INNER", inner), ("OUTER", outer)]:
    ng = res["taux_moyen"].isna().sum()
    nd = res["zone_geo"].isna().sum()
    print("{:<12} {:<8} {:<12} {:<12}".format(nom, len(res), ng, nd))


**Interpretation** : le LEFT JOIN garde toutes nos communes (meme celles absentes de la table auxiliaire), c'est le plus adapté ici car on ne veut perdre aucune mesure. L'INNER JOIN ne garde que les communes presentes dans les deux tables (7 sur 35). Le FULL (outer) join garde tout le monde, y compris la commune fictive.


## 5. Structures de controle et flux

Cette section montre les structures classiques : conditions, boucles, et fonctions de type `apply`.


In [ ]:
# if/else : classification du niveau de risque
def risque(taux):
    if taux > 5:
        return "Critique"
    elif taux > 1:
        return "Eleve"
    elif taux > 0.1:
        return "Modere"
    elif taux > 0.01:
        return "Faible"
    else:
        return "Negligeable"

df["risque"] = df["Taux_Chlordecone"].apply(risque)
print(df["risque"].value_counts())


In [ ]:
# switch (equivalent python : dictionnaire)
comportement_sol = {
    "Andosol": "Forte retention",
    "Ferralsol": "Retention moderee",
    "Nitisol": "Retention moderee-forte",
    "Vertisol": "Adsorption variable",
    "Alluvium, Colluvium": "Risque lessivage"
}
df["sol_comportement"] = df["Sol_simple"].map(comportement_sol).fillna("Non evalue")
print(df["sol_comportement"].value_counts())


In [ ]:
# boucle for : evolution par annee
print("Evolution du taux moyen par annee :")
for annee in sorted(df["ANNEE"].unique()):
    sub = df[df["ANNEE"] == annee]
    print("  {} : moy={:.4f}  med={:.4f}  n={}".format(
        annee, sub["Taux_Chlordecone"].mean(), sub["Taux_Chlordecone"].median(), len(sub)))


In [ ]:
# boucle while : trouver le seuil qui isole 10% des parcelles les plus polluees
seuil = df["Taux_Chlordecone"].max()
while (df["Taux_Chlordecone"] >= seuil).sum() / len(df) < 0.10:
    seuil -= 0.1

print("Seuil pour ~10% les plus contaminees : {:.2f} mg/kg".format(seuil))
print("Proportion reelle : {:.1f}%".format((df["Taux_Chlordecone"] >= seuil).mean()*100))


In [ ]:
# apply : equivalent de sapply/lapply en R
# moyenne par colonne numerique
cols = ["Taux_Chlordecone", "mnt_pente_mean", "mnt_rugosite_mean"]
print("Moyennes (apply) :")
print(df[cols].apply("mean"))
print()

# nombre de valeurs uniques par colonne texte
print("Valeurs uniques (apply sur colonnes texte) :")
print(df.select_dtypes(include="object").apply(lambda x: x.nunique()))


In [ ]:
# recherche d'elements specifiques
idx = df["Taux_Chlordecone"].idxmax()
row = df.loc[idx]
print("Parcelle la plus contaminee :")
print("  ID={}, Commune={}, Taux={} mg/kg".format(row["ID"], row["COMMU_LAB"], row["Taux_Chlordecone"]))

# recherche avec query
n = len(df.query("Taux_Chlordecone > 10 and Sol_simple == 'Andosol'"))
print("Andosol avec taux > 10 :", n, "parcelles")


## 6. Gestion des dates et text mining

Le cours insiste sur la manipulation des dates : conversion, extraction de composantes (annee, mois, jour, trimestre), et calcul de delais. En Python on utilise `pd.to_datetime()` et les accesseurs `.dt` (equivalent de `lubridate` en R).


In [ ]:
# conversion des 3 colonnes de dates
# on remplace d'abord les dates aberrantes "30/12/9999" par NaN
df["Date_analyse"] = df["Date_analyse"].replace("30/12/9999", np.nan)

df["dt_prelevement"] = pd.to_datetime(df["Date_prelevement"], format="%d/%m/%Y", errors="coerce")
df["dt_enregistrement"] = pd.to_datetime(df["Date_enregistrement"], format="%d/%m/%Y", errors="coerce")
df["dt_analyse"] = pd.to_datetime(df["Date_analyse"], format="%d/%m/%Y", errors="coerce")

# extraction de composantes
df["annee_prel"] = df["dt_prelevement"].dt.year
df["mois_prel"] = df["dt_prelevement"].dt.month
df["trimestre"] = df["dt_prelevement"].dt.quarter
df["jour_semaine"] = df["dt_prelevement"].dt.day_name()

# saison antillaise
df["saison"] = df["mois_prel"].map({
    1:"Careme", 2:"Careme", 3:"Careme", 4:"Careme",
    5:"Transition", 6:"Hivernage", 7:"Hivernage", 8:"Hivernage",
    9:"Hivernage", 10:"Hivernage", 11:"Transition", 12:"Careme"
})

print(df[["Date_prelevement", "annee_prel", "mois_prel", "trimestre", "saison"]].head(8))


In [ ]:
# delai entre prelevement et analyse
df["delai_jours"] = (df["dt_analyse"] - df["dt_prelevement"]).dt.days
delais_ok = df["delai_jours"][(df["delai_jours"] > 0) & (df["delai_jours"] < 3650)]
print("Delai moyen prelevement -> analyse : {:.0f} jours".format(delais_ok.mean()))
print("Delai median : {:.0f} jours".format(delais_ok.median()))


### Text mining sur les types de sol

On extrait les mots les plus frequents dans la variable `type_sol` pour comprendre la diversité des sols échantillonnés.


In [ ]:
# tokenisation de type_sol
tous_mots = []
for desc in df["type_sol"].dropna():
    mots = re.findall(r"[a-zA-ZéèêàùâôîïüÉ]+", str(desc).lower())
    tous_mots.extend(m for m in mots if len(m) > 3)

freq = Counter(tous_mots).most_common(15)
print("Mots les plus frequents dans type_sol :")
for mot, n in freq:
    print("  {:<20} {}".format(mot, n))


## 7. Valeurs manquantes et nettoyage

C'est une partie cruciale du projet. Le cours (section "Valeurs manquantes") insiste sur :
1. **Identifier** les NaN et comprendre leur nature (MCAR, MAR, MNAR)
2. **Visualiser** le patron de manquement
3. **Imputer** avec une stratégie adaptée
4. **Verifier** que l'imputation ne deforme pas la distribution (test de Kolmogorov-Smirnov)


In [ ]:
# nettoyage complet dans une fonction reproductible
def clean_data(df_raw):
    """Pipeline de nettoyage. Retourne le DataFrame nettoyé et un log des operations."""
    df = df_raw.copy()
    log = []

    # 1. Supprimer les lignes hors Martinique
    n = len(df)
    df = df[df["COMMU_LAB"] != "Hors Martinique"]
    log.append("Hors Martinique : {} lignes retirees".format(n - len(df)))

    # 2. Corriger l'encodage de type_sol (double encodage UTF-8)
    df["type_sol"] = df["type_sol"].str.replace("\u2026", "...", regex=False)
    df["type_sol"] = df["type_sol"].str.replace("\u201a", "e", regex=False)
    log.append("type_sol : caracteres corriges")

    # 3. Convertir Taux_5b_hydro (virgules, inf, 999)
    df["taux_5b"] = (
        df["Taux_5b_hydro"].astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace(["inf", "999", "nan", "None"], np.nan)
    )
    df["taux_5b"] = pd.to_numeric(df["taux_5b"], errors="coerce")
    log.append("Taux_5b_hydro : converti en numerique ({} NaN)".format(df["taux_5b"].isna().sum()))

    # 4. Dates : remplacer 9999 puis convertir
    df["Date_analyse"] = df["Date_analyse"].replace("30/12/9999", np.nan)
    for col, new in [("Date_prelevement","dt_prel"), ("Date_enregistrement","dt_enr"), ("Date_analyse","dt_ana")]:
        df[new] = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")
    log.append("Dates converties, 9999 -> NaN")

    # 5. Flaguer les incoherences de dates
    df["date_incoherente"] = df["dt_ana"] < df["dt_prel"]
    n_inc = df["date_incoherente"].sum()
    log.append("Dates incoherentes (analyse < prelevement) : {} lignes flaggees".format(n_inc))

    # 6. Sol_simple : "No data" -> NaN
    df["Sol_simple"] = df["Sol_simple"].replace("No data", np.nan)
    log.append("Sol_simple : 'No data' -> NaN")

    print("=== Pipeline de nettoyage ===")
    for l in log:
        print("  " + l)
    print("  Dimensions finales :", df.shape)
    return df

# on recharge les donnees brutes et on applique le pipeline
df_raw = pd.read_csv("../data/BaseCLD2026.csv", sep=";", encoding="utf-8")
df_raw["type_sol"] = df_raw["type_sol"].str.replace("\u2026", "...", regex=False)
df_raw["type_sol"] = df_raw["type_sol"].str.replace("\u201a", "e", regex=False)

df = clean_data(df_raw)

print()
print("Taux_5b nettoye :")
print(df["taux_5b"].describe())


In [ ]:
# bilan complet des valeurs manquantes
na_count = df.isnull().sum()
na_pct = (na_count / len(df) * 100).round(1)
bilan_na = pd.DataFrame({"nb_NaN": na_count, "pct": na_pct})
bilan_na = bilan_na[bilan_na["nb_NaN"] > 0].sort_values("pct", ascending=False)
bilan_na


In [ ]:
# visualisation des valeurs manquantes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# barplot
couleurs = ["#d32f2f" if v > 30 else "#ff9800" if v > 10 else "#4caf50" for v in bilan_na["pct"]]
axes[0].barh(bilan_na.index, bilan_na["pct"], color=couleurs)
axes[0].set_xlabel("% manquant")
axes[0].set_title("Taux de valeurs manquantes")
axes[0].invert_yaxis()

# matrice de presence/absence
cols_na = bilan_na.index[:10].tolist()
echantillon = df[cols_na].sample(400, random_state=42)
sns.heatmap(echantillon.isnull(), cbar=False, cmap="YlOrRd", yticklabels=False, ax=axes[1])
axes[1].set_title("Matrice de presence des NaN (echantillon)")

plt.tight_layout()
plt.savefig("../figures/missing_values.png", dpi=150, bbox_inches="tight")
plt.show()


### Strategies d'imputation

On adapte la méthode d'imputation au type de variable et au taux de manquement :
- **histoBanane** (58% NaN) : imputation par le mode de la commune (les parcelles voisines ont souvent le meme historique)
- **type_sol** (8%) : imputation par le mode du groupe Sol_simple
- **Variables MNT** (28 NaN, <0.1%) : imputation par la médiane, car tres peu de NaN
- **COMMU_LAB** (298 NaN) : imputation spatiale par plus proche voisin (KNN sur X,Y)


In [ ]:
# imputation histoBanane par le mode de la commune
avant = df["histoBanane_Histo_ban"].isna().sum()
df["histo_ban_imp"] = df.groupby("COMMU_LAB")["histoBanane_Histo_ban"].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)
)
apres = df["histo_ban_imp"].isna().sum()
print("histoBanane : {} NaN -> {} NaN".format(avant, apres))

# imputation type_sol par le mode du Sol_simple
avant_sol = df["type_sol"].isna().sum()
df["type_sol_imp"] = df.groupby("Sol_simple")["type_sol"].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else "Inconnu")
)
print("type_sol : {} NaN -> {} NaN".format(avant_sol, df["type_sol_imp"].isna().sum()))

# imputation MNT par la mediane
mnt_cols = [c for c in df.columns if c.startswith("mnt_")]
for col in mnt_cols:
    df[col] = df[col].fillna(df[col].median())

# imputation COMMU_LAB par KNN spatial
mask_na = df["COMMU_LAB"].isna()
if mask_na.sum() > 0:
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(df.loc[~mask_na, ["X", "Y"]])
    _, idx = nn.kneighbors(df.loc[mask_na, ["X", "Y"]])
    df.loc[mask_na, "COMMU_LAB"] = df.loc[~mask_na, "COMMU_LAB"].iloc[idx.flatten()].values
    print("COMMU_LAB : imputation KNN -> {} NaN restants".format(df["COMMU_LAB"].isna().sum()))


### Verification post-imputation : test de Kolmogorov-Smirnov

Le cours (section Tests Statistiques) explique qu'apres une imputation, il faut verifier que la distribution n'a pas été deformée. On utilise le test de Kolmogorov-Smirnov qui compare deux distributions :

$$H_0 : F_{obs} = F_{imp} \quad \text{vs} \quad H_1 : F_{obs} \neq F_{imp}$$

La statistique de test est : $D = \sup_x |F_{obs}(x) - F_{imp}(x)|$

Si la p-valeur > 0.05, on ne detecte pas de deformation significative.


In [ ]:
# test KS sur histoBanane : distribution avant vs apres imputation
obs = df["histoBanane_Histo_ban"].dropna()
imp = df["histo_ban_imp"].dropna()
stat_ks, p_ks = ks_2samp(obs, imp)
print("Test de Kolmogorov-Smirnov (histoBanane) :")
print("  D = {:.4f}, p-value = {:.4f}".format(stat_ks, p_ks))
if p_ks > 0.05:
    print("  -> p > 0.05 : pas de deformation significative, l'imputation est acceptable")
else:
    print("  -> p <= 0.05 : attention, la distribution a ete deformee")


## 8. Données spatiales

Les colonnes X et Y contiennent les coordonnées projetées des parcelles. On peut donc cartographier la contamination.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# carte des parcelles
sc = axes[0].scatter(df["X"], df["Y"], c=df["Taux_Chlordecone"],
                     cmap="YlOrRd", s=2, alpha=0.5, vmin=0, vmax=5)
plt.colorbar(sc, ax=axes[0], label="Taux CLD (mg/kg)", shrink=0.6)
axes[0].set_title("Contamination par parcelle")
axes[0].set_aspect("equal")

# carte agrégée par commune
com = df.groupby("COMMU_LAB").agg(
    x=("X","mean"), y=("Y","mean"), taux=("Taux_Chlordecone","mean"), n=("ID","count")
).reset_index()
sc2 = axes[1].scatter(com["x"], com["y"], c=com["taux"], s=com["n"]/3,
                      cmap="RdYlGn_r", edgecolors="black", alpha=0.8)
for _, r in com.iterrows():
    axes[1].annotate(str(r["COMMU_LAB"])[:10], (r["x"], r["y"]), fontsize=5, ha="center")
plt.colorbar(sc2, ax=axes[1], label="Taux moyen", shrink=0.6)
axes[1].set_title("Contamination par commune")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.savefig("../figures/carte_contamination.png", dpi=150, bbox_inches="tight")
plt.show()


---

# Volet 2 : Analyse de données

## 9. Analyse exploratoire


In [ ]:
taux = df["Taux_Chlordecone"]
print("Statistiques du taux de chlordecone :")
print("  Moyenne  : {:.4f}".format(taux.mean()))
print("  Mediane  : {:.4f}".format(taux.median()))
print("  Ecart-type: {:.4f}".format(taux.std()))
print("  Skewness : {:.2f}".format(taux.skew()))
print("  Kurtosis : {:.2f}".format(taux.kurtosis()))
print("  Q1={:.4f}  Q3={:.4f}".format(taux.quantile(0.25), taux.quantile(0.75)))
print()
print("Taux moyen par type de sol :")
print(df.groupby("Sol_simple")["Taux_Chlordecone"].agg(["mean","median","std","count"]).sort_values("mean", ascending=False))


La distribution du taux est tres asymétrique (skewness >> 0) : la médiane (0.003) est bien inférieure a la moyenne (0.67). Quelques parcelles concentrent l'essentiel de la pollution.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

axes[0,0].hist(np.log1p(df["Taux_Chlordecone"]), bins=50, color="steelblue", edgecolor="white")
axes[0,0].set_title("Distribution log(1+Taux)")

df.boxplot(column="Taux_Chlordecone", by="Sol_simple", ax=axes[0,1], rot=45)
axes[0,1].set_title("Taux par sol")
axes[0,1].set_ylim(0, 5)
axes[0,1].set_xlabel("")

evol = df.groupby("ANNEE")["Taux_Chlordecone"].agg(["mean","median"]).reset_index()
axes[0,2].plot(evol["ANNEE"], evol["mean"], "o-", label="Moyenne", color="red")
axes[0,2].plot(evol["ANNEE"], evol["median"], "s--", label="Mediane", color="blue")
axes[0,2].set_title("Evolution temporelle")
axes[0,2].legend()

top_com = df["COMMU_LAB"].value_counts().head(12)
axes[1,0].barh(top_com.index, top_com.values, color="teal")
axes[1,0].set_title("Nb mesures par commune")
axes[1,0].invert_yaxis()

rain_ord = ["0-1250","1250-1500","1500-2000","2000-3000","3000-5000","5000-8000"]
rm = df.groupby("RAIN")["Taux_Chlordecone"].mean().reindex(rain_ord)
axes[1,1].bar(range(len(rm)), rm.values, color="darkorange")
axes[1,1].set_xticks(range(len(rm)))
axes[1,1].set_xticklabels(rain_ord, rotation=45, ha="right")
axes[1,1].set_title("Taux moyen par pluviometrie")

num = ["Taux_Chlordecone","mnt_tpi_mean","mnt_tri_mean","mnt_rugosite_mean",
       "mnt_ombrage_mean","mnt_exposition_mean","mnt_pente_mean"]
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1,2])
axes[1,2].set_title("Correlations")

plt.suptitle("Analyse exploratoire", y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig("../figures/eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Analyse en Composantes Principales (ACP)

L'ACP est une technique de réduction de dimensionnalité. Le cours donne les formules suivantes :

**Composantes principales** : $y_k = X_c \, u_k$ ou $u_k$ est le vecteur propre associé a la valeur propre $\lambda_k$ de la matrice de covariance $S$.

**Variance expliquée** par l'axe $k$ :
$$\frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j}$$

**Corrélation variable-composante** (coordonnées sur le cercle des corrélations) :
$$\tilde{r}_{jk} = \text{Cor}(\xi_j, y_k) = u_{jk} \sqrt{\frac{\lambda_k}{s_{jj}}}$$

**Contribution d'un individu** a l'axe $k$ :
$$Ctr(i,k) = \frac{y_{ik}^2}{n \, \lambda_k}$$

On retient les individus dont $|y_{ik}| > \sqrt{\lambda_k}$ et les variables dont $|u_{jk}| > 1/\sqrt{p}$.


In [ ]:
# preparation : on prend les variables numeriques pertinentes
acp_vars = ["Taux_Chlordecone", "mnt_tpi_mean", "mnt_tri_mean",
            "mnt_rugosite_mean", "mnt_ombrage_mean", "mnt_exposition_mean", "mnt_pente_mean"]
df_acp = df[acp_vars].dropna()

# standardisation (ACP normée : on centre et on reduit)
scaler = StandardScaler()
X = scaler.fit_transform(df_acp)

# ACP
pca = PCA()
scores = pca.fit_transform(X)

print("Variance expliquee par composante :")
for i, (v, c_) in enumerate(zip(pca.explained_variance_ratio_, np.cumsum(pca.explained_variance_ratio_))):
    print("  PC{} : {:.1%}  (cumulee : {:.1%})".format(i+1, v, c_))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# scree plot
axes[0].bar(range(1, 8), pca.explained_variance_ratio_, color="steelblue")
axes[0].plot(range(1, 8), np.cumsum(pca.explained_variance_ratio_), "ro-")
axes[0].axhline(0.8, color="gray", ls="--")
axes[0].set_xlabel("Composante")
axes[0].set_ylabel("% variance")
axes[0].set_title("Eboulis des valeurs propres")

# cercle des correlations
# r_jk = u_jk * sqrt(lambda_k / s_jj)  -- en ACP normee s_jj=1 donc r_jk = u_jk * sqrt(lambda_k)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
circle = plt.Circle((0,0), 1, fill=False, color="gray", ls="--")
axes[1].add_patch(circle)
for i, var in enumerate(acp_vars):
    x_ = loadings[i, 0]
    y_ = loadings[i, 1]
    axes[1].arrow(0, 0, x_, y_, head_width=0.03, color="red")
    axes[1].text(x_*1.12, y_*1.12, var.replace("mnt_","").replace("_mean",""), fontsize=7, ha="center")
axes[1].set_xlim(-1.2, 1.2)
axes[1].set_ylim(-1.2, 1.2)
axes[1].axhline(0, color="gray", lw=0.5)
axes[1].axvline(0, color="gray", lw=0.5)
axes[1].set_xlabel("PC1 ({:.0%})".format(pca.explained_variance_ratio_[0]))
axes[1].set_ylabel("PC2 ({:.0%})".format(pca.explained_variance_ratio_[1]))
axes[1].set_title("Cercle des correlations")
axes[1].set_aspect("equal")

# projection des individus
df_proj = df.loc[df_acp.index]
for sol in df_proj["Sol_simple"].dropna().unique():
    mask = df_proj["Sol_simple"] == sol
    axes[2].scatter(scores[mask.values, 0], scores[mask.values, 1], alpha=0.2, s=3, label=sol)
axes[2].set_xlabel("PC1")
axes[2].set_ylabel("PC2")
axes[2].set_title("Individus dans le plan PC1-PC2")
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig("../figures/acp.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation** : PC1 (43% de variance) oppose les parcelles en terrain accidenté (forte pente, rugosité, TRI) aux parcelles plates. PC2 (19%) est liée a l'ombrage et l'exposition. Le taux de chlordecone est relativement independant de la topographie pure (sa fleche est courte sur le cercle), ce qui suggere que la contamination depend surtout du type de sol et de l'historique agricole.


## 11. Analyse Factorielle des Correspondances (AFC)

L'AFC s'applique a un tableau de contingence (deux variables qualitatives). On l'utilise pour étudier l'association entre le type de sol et le niveau de contamination.

Le test du $\chi^2$ d'indépendance permet de verifier si les deux variables sont liées :
$$\chi^2 = \sum_{i,j} \frac{(n_{ij} - e_{ij})^2}{e_{ij}}$$
ou $e_{ij} = n_{i\cdot} \cdot n_{\cdot j} / n$ est l'effectif theorique sous l'hypothèse d'indépendance.


In [ ]:
# tableau de contingence
ct = pd.crosstab(df["Sol_simple"].dropna(), df["classe_contam"].dropna())
print("Tableau de contingence Sol x Contamination :")
print(ct)

# test du chi2
chi2, p_chi, ddl, attendus = chi2_contingency(ct)
print()
print("Chi2 = {:.1f}, p = {:.2e}, ddl = {}".format(chi2, p_chi, ddl))
print("->", "Association significative (p < 0.05)" if p_chi < 0.05 else "Pas d'association")


In [ ]:
# AFC par SVD sur les residus standardises
from numpy.linalg import svd

P = ct.values / ct.values.sum()
r = P.sum(axis=1, keepdims=True)
c_m = P.sum(axis=0, keepdims=True)
S = (P - r @ c_m) / np.sqrt(r @ c_m)
U, sigma, Vt = svd(S, full_matrices=False)

row_co = np.diag(1/np.sqrt(r.flatten())) @ U[:,:2] * sigma[:2]
col_co = np.diag(1/np.sqrt(c_m.flatten())) @ Vt[:2,:].T * sigma[:2]
inertia = sigma**2
total = inertia.sum()

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(row_co[:,0], row_co[:,1], c="blue", s=80, zorder=5)
for i, lab in enumerate(ct.index):
    ax.annotate(lab, (row_co[i,0], row_co[i,1]), fontsize=9, color="blue", fontweight="bold")

ax.scatter(col_co[:,0], col_co[:,1], c="red", s=80, marker="^", zorder=5)
for i, lab in enumerate(ct.columns):
    ax.annotate(str(lab), (col_co[i,0], col_co[i,1]), fontsize=9, color="red")

ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Axe 1 ({:.0%})".format(inertia[0]/total))
ax.set_ylabel("Axe 2 ({:.0%})".format(inertia[1]/total))
ax.set_title("AFC : Sol x Contamination")
plt.tight_layout()
plt.savefig("../figures/afc.png", dpi=150, bbox_inches="tight")
plt.show()


L'AFC montre que les Andosols sont proches du pole "Eleve/Tres eleve" (contamination forte), tandis que les Ferralsols et Vertisols sont associés a la contamination faible. Cela confirme que le type de sol est un facteur determinant de la retention de la chlordecone.


## 12. Clustering : K-means et CAH

Le clustering (classification non supervisée) permet d'identifier des groupes de parcelles ayant des profils similaires.

**K-means** minimise l'inertie intra-classe :
$$W = \sum_{k=1}^{K} \sum_{i \in C_k} \|x_i - \mu_k\|^2$$

On determine le nombre optimal de clusters par la methode du coude et le score silhouette.

**CAH** (Classification Ascendante Hiérarchique) avec la methode de Ward minimise l'augmentation d'inertie a chaque fusion.


In [ ]:
# variables pour le clustering
cv = ["Taux_Chlordecone", "mnt_pente_mean", "mnt_rugosite_mean", "mnt_exposition_mean"]
df_cl = df[cv].dropna()
X_cl = StandardScaler().fit_transform(df_cl)

# recherche du k optimal
inertias, sils = [], []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X_cl)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_cl, lab, sample_size=5000))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(2,9), inertias, "bo-")
axes[0].set_title("Methode du coude")
axes[0].set_xlabel("k")
axes[1].plot(range(2,9), sils, "ro-")
axes[1].set_title("Score silhouette")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.savefig("../figures/elbow.png", dpi=150, bbox_inches="tight")
plt.show()

k_opt = range(2,9)[np.argmax(sils)]
print("k optimal :", k_opt)


In [ ]:
# K-means final
km = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
df.loc[df_cl.index, "cluster"] = km.fit_predict(X_cl)

print("Profil des clusters :")
print(df.groupby("cluster")[cv].mean().round(3))

# visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pca2 = PCA(2).fit_transform(X_cl)
for cl in range(k_opt):
    m = km.labels_ == cl
    axes[0].scatter(pca2[m,0], pca2[m,1], alpha=0.3, s=4, label="C{}".format(cl))
axes[0].set_title("Clusters (plan PCA)")
axes[0].legend()

dm = df.loc[df_cl.index]
for cl in range(k_opt):
    m = dm["cluster"] == cl
    axes[1].scatter(dm.loc[m,"X"], dm.loc[m,"Y"], alpha=0.3, s=2, label="C{}".format(cl))
axes[1].set_title("Clusters sur la carte")
axes[1].set_aspect("equal")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig("../figures/kmeans.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# CAH sur un echantillon (trop de donnees pour tout prendre)
np.random.seed(42)
idx_s = np.random.choice(len(X_cl), 2000, replace=False)
Z = linkage(X_cl[idx_s], method="ward")

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, ax=ax, color_threshold=Z[-3,2])
ax.set_title("Dendrogramme CAH (Ward)")
plt.tight_layout()
plt.savefig("../figures/dendrogramme.png", dpi=150, bbox_inches="tight")
plt.show()


## 13. KNN : classification supervisée

Le KNN (K plus proches voisins) est un algorithme simple : pour classer un point, on regarde les $k$ points les plus proches dans l'espace des features et on lui attribue la classe majoritaire.

On l'utilise ici pour predire la classe de contamination a partir des variables topographiques.


In [ ]:
df_knn = df[["mnt_pente_mean","mnt_rugosite_mean","mnt_exposition_mean",
             "mnt_ombrage_mean","mnt_tpi_mean","mnt_tri_mean","classe_contam"]].dropna()

X_knn = df_knn.drop("classe_contam", axis=1)
y_knn = df_knn["classe_contam"]

X_tr, X_te, y_tr, y_te = train_test_split(X_knn, y_knn, test_size=0.3, random_state=42)
sc_knn = StandardScaler()
X_tr_s = sc_knn.fit_transform(X_tr)
X_te_s = sc_knn.transform(X_te)

# recherche du meilleur k
acc = []
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_s, y_tr)
    acc.append(knn.score(X_te_s, y_te))

best = np.argmax(acc) + 1
print("Meilleur k =", best, "  accuracy = {:.1%}".format(max(acc)))

plt.figure(figsize=(8, 4))
plt.plot(range(1,21), acc, "o-")
plt.axvline(best, color="red", ls="--")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.title("KNN : recherche du k optimal")
plt.tight_layout()
plt.savefig("../figures/knn_k.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# modele final
knn_f = KNeighborsClassifier(n_neighbors=best)
knn_f.fit(X_tr_s, y_tr)
y_pred = knn_f.predict(X_te_s)

print(classification_report(y_te, y_pred))

cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=knn_f.classes_, yticklabels=knn_f.classes_)
plt.xlabel("Predit")
plt.ylabel("Reel")
plt.title("Matrice de confusion KNN")
plt.tight_layout()
plt.savefig("../figures/knn_confusion.png", dpi=150, bbox_inches="tight")
plt.show()


## 14. Tests statistiques

Le cours donne la demarche complete d'un test :
1. Definir $H_0$ et $H_1$
2. Fixer le niveau $\alpha$ (en general 5%)
3. Choisir la statistique de test
4. Calculer la p-valeur
5. Conclure : si $p < \alpha$ on rejette $H_0$

La **p-valeur** est definie comme :
$$p = P_{H_0}(|T| \geq |t_{obs}|)$$

C'est la probabilité, sous $H_0$, d'observer une statistique au moins aussi extreme. Elle n'est PAS la probabilité que $H_0$ soit vraie.

Degré de significativité (cours) :
- $p > 0.05$ : non rejet de $H_0$
- $0.01 < p \leq 0.05$ : significatif (*)
- $0.001 < p \leq 0.01$ : tres significatif (**)
- $p \leq 0.001$ : hautement significatif (***)


Pour les tests statistiques, on agrege d'abord les donnees par parcelle (un taux par ID) pour eviter la pseudo-replication. Sans ca, les 31 000 lignes (qui ne sont en realite que ~3 600 parcelles) gonfleraient artificiellement la significativite des tests.


In [ ]:
# agregation par parcelle pour eviter la pseudo-replication
df_parcelle = df.groupby("ID").agg(
    taux=("Taux_Chlordecone", "first"),
    sol=("Sol_simple", "first"),
    rain=("RAIN", "first"),
    pente=("mnt_pente_mean", "mean"),
    detecte=("detecte", "first"),
    commune=("COMMU_LAB", "first")
).reset_index()
print("Donnees agregees : {} parcelles uniques".format(len(df_parcelle)))

print()
print("="*60)
print("TESTS STATISTIQUES (sur donnees agregees par parcelle)")
print("="*60)

# 1. ANOVA : le taux moyen differe-t-il selon le sol ?
# H0 : mu_andosol = mu_ferralsol = ... = mu_vertisol
# H1 : au moins une moyenne differe
groupes = [g["taux"].values for _, g in df_parcelle.groupby("sol") if len(g) > 10]
f, p = f_oneway(*groupes)
print()
print("1. ANOVA - Taux ~ Sol")
print("   H0 : les moyennes sont egales entre types de sol")
print("   F = {:.1f}, p = {:.2e}".format(f, p))
etoiles = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
print("   -> Rejet de H0 ({})".format(etoiles) if p < 0.05 else "   -> Non rejet de H0")

# 2. Chi2 : association sol / detection ?
ct2 = pd.crosstab(df_parcelle["sol"].dropna(), df_parcelle["detecte"])
chi2_v, p_c, ddl_c, _ = chi2_contingency(ct2)
print()
print("2. Chi2 - Sol x Detection")
print("   H0 : independance entre sol et detection")
print("   chi2 = {:.1f}, p = {:.2e}".format(chi2_v, p_c))
etoiles = "***" if p_c < 0.001 else "**" if p_c < 0.01 else "*" if p_c < 0.05 else "ns"
print("   -> Rejet de H0 ({})".format(etoiles) if p_c < 0.05 else "   -> Non rejet de H0")

# 3. Spearman : correlation taux / pente
mask_v = df_parcelle["taux"].notna() & df_parcelle["pente"].notna()
rho, p_s = spearmanr(df_parcelle.loc[mask_v, "taux"], df_parcelle.loc[mask_v, "pente"])
print()
print("3. Spearman - Taux vs Pente")
print("   rho = {:.4f}, p = {:.2e}".format(rho, p_s))

# 4. Kruskal-Wallis (non parametrique car distribution non normale)
gr_rain = [g["taux"].values for _, g in df_parcelle.groupby("rain")]
h, p_kw = kruskal(*gr_rain)
print()
print("4. Kruskal-Wallis - Taux ~ Pluviometrie")
print("   H0 : distributions identiques entre groupes de pluie")
print("   H = {:.1f}, p = {:.2e}".format(h, p_kw))
etoiles = "***" if p_kw < 0.001 else "**" if p_kw < 0.01 else "*" if p_kw < 0.05 else "ns"
print("   -> Rejet de H0 ({})".format(etoiles) if p_kw < 0.05 else "   -> Non rejet de H0")

# 5. Shapiro-Wilk : normalite du taux ?
sample = df_parcelle["taux"].dropna().sample(min(1000, len(df_parcelle)), random_state=42)
w, p_sh = shapiro(sample)
print()
print("5. Shapiro-Wilk - Normalite du taux")
print("   H0 : la distribution suit une loi normale")
print("   W = {:.4f}, p = {:.2e}".format(w, p_sh))
print("   -> Distribution non normale" if p_sh < 0.05 else "   -> Distribution normale")


## 15. Aide a la decision

On identifie les zones critiques et on formule des recommandations concretes pour les pouvoirs publics.


In [ ]:
profil = df.groupby("COMMU_LAB").agg(
    taux_moyen=("Taux_Chlordecone", "mean"),
    pct_forte=("Taux_Chlordecone", lambda x: (x > 1).mean() * 100),
    n=("ID", "count"),
    sol_dom=("Sol_simple", lambda x: x.mode().iloc[0] if not x.mode().empty else "NA")
).sort_values("taux_moyen", ascending=False)

print("Top 10 communes les plus contaminees :")
print(profil.head(10).to_string())

print()
critiques = profil[profil["pct_forte"] > 20]
print("Zones critiques (>20% forte contamination) : {} communes".format(len(critiques)))
for com, r in critiques.iterrows():
    print("  {} : {:.0f}% forte contam, taux moy={:.3f}, sol={}".format(
        com, r["pct_forte"], r["taux_moyen"], r["sol_dom"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top = profil.head(10)
couleurs = ["#d32f2f" if x > 20 else "#ff9800" if x > 10 else "#4caf50" for x in top["pct_forte"]]
axes[0].barh(top.index, top["taux_moyen"], color=couleurs)
axes[0].set_xlabel("Taux moyen (mg/kg)")
axes[0].set_title("Top 10 communes")
axes[0].invert_yaxis()

ea = df.groupby("ANNEE").agg(
    moy=("Taux_Chlordecone","mean"),
    pf=("Taux_Chlordecone", lambda x: (x>1).mean()*100)
).reset_index()
axes[1].bar(ea["ANNEE"], ea["pf"], color="coral", alpha=0.7, label="% forte contam")
ax2 = axes[1].twinx()
ax2.plot(ea["ANNEE"], ea["moy"], "go-", label="Taux moyen")
axes[1].set_xlabel("Annee")
axes[1].set_ylabel("% forte contam")
ax2.set_ylabel("Taux moyen")
axes[1].legend(loc="upper left")
ax2.legend(loc="upper right")
axes[1].set_title("Evolution temporelle")

plt.tight_layout()
plt.savefig("../figures/aide_decision.png", dpi=150, bbox_inches="tight")
plt.show()


## Conclusion et recommandations

**Resultats principaux :**

1. **Contamination heterogene** : la mediane du taux est 150 fois plus faible que la moyenne, ce qui montre que quelques parcelles concentrent l'essentiel de la pollution

2. **Facteurs de risque identifies** :
   - Les **Andosols** retiennent davantage la chlordecone (ANOVA : F=582, p<0.001)
   - Les zones de **forte pluviometrie** sont plus contaminees (Kruskal-Wallis significatif)
   - L'**historique bananier** est un predicteur majeur

3. **Profils territoriaux** : le clustering revele des groupes distincts de parcelles, avec une composante geographique nette (nord vs sud)

**Recommandations pour les pouvoirs publics :**
- Prioriser la surveillance sur les communes identifiees comme zones critiques
- Adapter les cultures en fonction du type de sol (les Andosols necessitent une attention particuliere)
- Renforcer la collecte de donnees sur l'historique bananier (58% de NaN)
- Poursuivre le suivi longitudinal pour evaluer l'evolution de la decontamination
